In [13]:
import xarray as xr
import numpy as np
import pandas as pd
from libpysal.weights import lat2W
from esda import Moran
np.random.seed(12345)

In [14]:
def extract_year_slice(ds, varname, year):
    da = ds[varname].sel(time=f"{year}-01-01")
    return da.values

In [15]:
def compute_moran(arr2d):
    H, W = arr2d.shape
    
    # NaN 填零（libpysal 需要）
    flat = np.nan_to_num(arr2d.flatten(), nan=0.0)
    
    w = lat2W(H, W)
    w.transform = "r"
    
    mi = Moran(flat, w, permutations=0)
    return mi.I


In [16]:
def compute_all_variables(ds, years, output_csv):
    varnames = list(ds.data_vars)  # 自动遍历所有变量
    results = []

    for year in years:
        print(f"Processing year {year} ...")
        row = {"year": year}

        for varname in varnames:
            print(f"started {varname}")
            arr2d = extract_year_slice(ds, varname, year)

            row[varname] = compute_moran(arr2d)

        results.append(row)

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"Saved CSV to {output_csv}")

    return df


In [17]:
years = list(range(2010, 2101, 10))
years.insert(0, 2005)

name = ["agri","forest","grassland"]
mode = ["basin","region"]

for m in mode:
    for n in name:
        print(f"calculating {m}_{n}_17regions.nc")
        ds = xr.open_dataset(f"../../NC/{m}_{n}_17regions.nc")

        df = compute_all_variables(ds, years, output_csv=f"../../CSV/moran/{m}_{n}_moran.csv")
        df


calculating basin_agri_17regions.nc
Processing year 2005 ...
started BRA_basin_agri
started CAN_basin_agri
started CHN_basin_agri
started CIS_basin_agri
started IND_basin_agri
started JPN_basin_agri
started TUR_basin_agri
started USA_basin_agri
started XAF_basin_agri
started XE25_basin_agri
started XER_basin_agri
started XLM_basin_agri


KeyboardInterrupt: 